# Foundation Models hook-up

Anton Antonov  
November 2026

---

# Introduction

This notebook demonstrates how to use Apple's [Foundation Models](https://developer.apple.com/documentation/foundationmodels) (FM) via the package ["WWW::LLaMA"](https://github.com/antononcube/Raku-WWW-LLaMA).
Both FM and "WWW::LLaMA" are compatible with OpenAI's API. Hence, the latter can be used for LLM-generation with the former.

---

## Setup

Load packages:

In [147]:
use WWW::LLaMA;
use WWW::LLaMA::Models;

use LLM::Functions;
use LLM::Prompts;

use Data::Reshapers;
use Math::SparseMatrix;

In macOS 27+ run Apple's built-in Foundation Model (FM) with the command:

```sh
fm serve
```

Which will give the output:

```text
Foundation Models Serve
  url    http://127.0.0.1:1976
  access loopback-only

  · POST /v1/chat/completions
  · GET  /v1/models
  · GET  /health
```

Alternatively, start with a custom port / host / Unix socket:

```sh
fm serve --port 1976
fm serve --host 127.0.0.1 --port 1976
fm serve --socket /tmp/fm.sock
```

### Adding new LLaMA models

The models names "system" and "pcc" are recognized by "WWW::LLaMA:ver<1.0.5+>". In general, though, recognized model names can be added in the following way:

```raku
llama-known-models()<system> = True;
llama-known-models()<pcc> = True;
say llama-known-models();

llama-end-point-to-models()<chat/completions> = <gpt-3.5-turbo llamafile llama llava system pcc>;
say llama-end-point-to-models();
```

---

## Generation via subs of "WWW::LLaMA"

Set up the end point:

In [12]:
llama-base-url('http://127.0.0.1:1976')

http://127.0.0.1:1976

Generate a response to question with `llama-playground`:

In [ ]:
'What is the speed of a rocket leaving the Moon?'
==> llama-playground(path => 'chat/completion', model => 'system', format => 'values');

The speed of a rocket leaving the Moon depends on several factors, including the rocket's design, the mission's objectives, and the specific trajectory. Generally, a rocket leaving the lunar surface might reach speeds around 2,800 miles per hour (4,500 kilometers per hour) depending on the propulsion system and mission specifics. For precise calculations, consult mission reports or engineers specializing in lunar missions.

**Remark:** Alternatively, the option `base-url => 'http://127.0.0.1:1976'` can be given to `llama-playground`.

---

## Generations with the LLM-functions framework

### LLM configuration

Here is a corresponding LLM-configuration (see ["LLM::Functions"](https://github.com/antononcube/Raku-LLM-Functions)):

In [77]:
my $conf-fm = llm-configuration('LLaMA', model => 'system', base-url => 'http://127.0.0.1:1976', :4096max-tokens, temperature => 0.4);

LLM::Configuration(:name("llama"), :model("system"), :module("WWW::LLaMA"), :max-tokens(4096))

### Simple LLM generation requests

Generate response with FM using `llm-synthesize`:

In [72]:
llm-synthesize([
    'How mane people live in Peru?',
    ],
    e => $conf-fm
)

Approximately 33 million people live in Peru.

Raku code generation is not that good:

In [73]:
llm-synthesize([
    llm-prompt('CodeWriterX')('Raku'),
    'Iterate over a list of random reals.',
    ],
    e => $conf-fm
    )

```rakudo
@list = 0..1.random(5);
say @list;
```

### Complex queries

Currently, FM produces good results for simple queries, but for more complicated one the results are "approximately" good. For example:

In [93]:
my &ctFunc = 
llm-function( -> $what {[
    "Crows tabulate the $what and their time zones.",
    llm-prompt('NothingElse')('Markdown')
    ]},
    e => $conf-fm
)

LLM::Function(-> **@args, *%args { #`(Block|5155558486952) ... }, 'llama')

In [94]:
#% markdown
my $res = &ctFunc('top 10 largest by area countries')

| Country          | Area (sq km) | Time Zone       |
|------------------|--------------|-----------------|
| Russia           | 17,098,242   | UTC+2           |
| Canada           | 9,984,670    | UTC-5           |
| China            | 9,596,961    | UTC+8           |
| United States    | 9,524,275    | UTC-6           |
| Brazil           | 8,515,767    | UTC-3           |
| Australia        | 7,692,024    | UTC+10          |
| India            | 3,287,263    | UTC+5:30        |
| Argentina        | 2,780,400    | UTC-3           |
| Kazakhstan       | 2,724,900    | UTC+6           |
| Algeria          | 2,381,741    | UTC+1           |

That, of course, can be remedied by using more complicated prompts and additional transformations:

In [142]:
my &ctFunc2 = 
llm-function( -> $what {[
    "Cross-tabulate $what and their time zones.",
    'Give a JSON array of dictionaries, all having the same keys and at least one being "time-zone".',
    'Please, use short names for the time zones.',
    'Numeric results should be computable,', 
    'Do it quickly -- do not overthink it.',
    llm-prompt('NothingElse')('JSON')
    ]},
    e => $conf-fm,
    form => sub-parser('JSON'):drop
)

LLM::Function(-> **@args, *%args { #`(Block|5155926675992) ... }, 'llama')

In [143]:
my @data = |&ctFunc2('Top 10 largest by area countries');
deduce-type(@data)


Vector(Struct([area, country, time-zone], [Int, Str, Str]), 10)

Here is the obtained result:

In [144]:
#% html
my @field-names = <country area time-zone>;
@data
==> to-html(:@field-names, align => 'left')

country,area,time-zone
Russia,170982426,UTC+2
Canada,99846700,UTC-5
China,93893411,UTC+8
United States,98335170,UTC-6
Brazil,85157673,UTC-3
Australia,76920242,UTC+10
India,32872636,UTC+5:30
Argentina,27804000,UTC-3
Kazakhstan,27249000,UTC+6
Algeria,23817485,UTC+1


Again, that result is table, not a contingency table, i.e. cross-tabulated result.
Here is the cross tabulation using `cross-tabulate` of ["Data::Reshapers"](https://github.com/antononcube/Raku-Data-Reshapers) given as a sparse matrix (see ["Math::SparseMatrix"](https://github.com/antononcube/Raku-Math-SparseMatrix)):

In [145]:
my $mat = cross-tabulate(@data, <country>, <time-zone>, :sparse);
$mat.print

––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––
                UTC+1    UTC+10   UTC+2    UTC+5:30 UTC+6    UTC+8    UTC-3    UTC-5    UTC-6   
––––––––––––––┼–––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––
Algeria       │ 1        .        .        .        .        .        .        .        .       
Argentina     │ .        .        .        .        .        .        1        .        .       
Australia     │ .        1        .        .        .        .        .        .        .       
Brazil        │ .        .        .        .        .        .        1        .        .       
Canada        │ .        .        .        .        .        .        .        1        .       
China         │ .        .        .        .        .        1        .        .        .       
India         │ .        .        .        1        .        .        .        .        .       
Kazakhstan    │ .        .    